# Week 13 Lab: Unsupervised Learning (Clustering and PCA)

<a href="https://colab.research.google.com/github/bradleyboehmke/uc-bana-4080/blob/main/labs/lab-13-clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This lab introduces you to two fundamental unsupervised learning techniques: **K-Means clustering** and **Principal Component Analysis (PCA)**. Unlike supervised learning where we have labeled data, unsupervised learning discovers hidden patterns and structures in data without predefined outcomes.

In Part A, you'll work through guided examples with customer segmentation data and housing data to understand how clustering groups similar observations together and how PCA reduces dimensionality. In Part B, you'll complete three homework challenges aligned with this week's online quiz, applying these techniques to real-world datasets.

## 🎯 Learning Objectives
By the end of this lab, you will be able to:
- Apply K-Means clustering to segment observations into meaningful groups
- Use the elbow method and silhouette scores to determine optimal cluster numbers
- Interpret cluster characteristics and understand what makes each group unique
- Execute the complete PCA workflow: standardize, fit, transform, and interpret
- Use scree plots to decide how many principal components to retain
- Interpret component loadings to understand what each PC represents
- Combine PCA with machine learning models to improve performance and reduce complexity

## 📚 This Lab Reinforces
- **Chapter 31: Unsupervised Learning and Clustering**
- **Chapter 32: Dimension Reduction with PCA**
- **This week's homework quiz questions**

## 🕐 Estimated Time & Structure
**Total Time:** 65-70 minutes  
**Mode:** Individual work recommended for Part B (homework)

- **[0–30 min]** Part A: Guided Lab Practice (clustering and PCA walkthroughs with "Your Turn" exercises)
- **[30–70 min]** Part B: Homework Challenges (complete the three challenges that directly correspond to quiz questions)

You are encouraged to work in small groups of **2–4 students** during Part A to discuss concepts and approaches.

## 💡 Why This Matters
Unsupervised learning is essential when you don't have labeled data or want to discover hidden patterns. Businesses use clustering to segment customers, identify fraud patterns, and group similar products. PCA helps data scientists visualize high-dimensional data, speed up machine learning models, and remove redundant features. These techniques are foundational for exploratory data analysis, feature engineering, and building production ML systems.

## Setup

Run this cell to import all necessary libraries and set random seeds for reproducibility.

In [ ]:
# Required imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import silhouette_score, accuracy_score, homogeneity_score, completeness_score

# One random state for every split, KMeans, PCA, and model
RANDOM_STATE = 42

print("✓ Libraries imported successfully!")

---

# Part A: Guided Lab Practice (~30 min)

This section walks you through K-Means clustering and PCA with hands-on examples and "Your Turn" exercises.

## Section 1: Introduction to K-Means Clustering

**K-Means clustering** is an algorithm that groups similar observations into k clusters. The algorithm:
1. Randomly places k cluster centers
2. Assigns each point to its nearest center
3. Updates centers to the mean of assigned points
4. Repeats steps 2-3 until convergence

### Example: Customer Segmentation

Imagine you're a retail analyst with customer data showing annual income and spending scores. You want to segment customers for targeted marketing.

In [ ]:
# Create sample customer data
np.random.seed(42)
n_customers = 200

# Generate three natural customer segments
# Segment 1: High income, high spending
seg1_income = np.random.normal(80, 10, 50)
seg1_spending = np.random.normal(75, 10, 50)

# Segment 2: Low income, low spending
seg2_income = np.random.normal(30, 8, 75)
seg2_spending = np.random.normal(25, 8, 75)

# Segment 3: Medium income, high spending
seg3_income = np.random.normal(50, 10, 75)
seg3_spending = np.random.normal(65, 10, 75)

# Combine into dataframe
customers = pd.DataFrame({
    'Income': np.concatenate([seg1_income, seg2_income, seg3_income]),
    'Spending_Score': np.concatenate([seg1_spending, seg2_spending, seg3_spending])
})

print(f"Customer data shape: {customers.shape}")
customers.head()

In [ ]:
# Visualize the customer data
plt.figure(figsize=(8, 6))
plt.scatter(customers['Income'], customers['Spending_Score'], alpha=0.6, s=60)
plt.xlabel('Annual Income ($k)')
plt.ylabel('Spending Score (1-100)')
plt.title('Customer Income vs Spending (Unlabeled Data)')
plt.grid(True, alpha=0.3)
plt.show()

print("💡 Notice: We can visually see some natural groupings, but K-Means will find them algorithmically.")

### Step 1: Standardize the Data

K-Means is sensitive to scale. We need to standardize features so they contribute equally to distance calculations.

In [ ]:
# Standardize the features
scaler = StandardScaler()
customers_scaled = scaler.fit_transform(customers)

print(f"Original data shape: {customers.shape}")
print(f"Scaled data shape: {customers_scaled.shape}")
print(f"\nScaled data mean: {customers_scaled.mean():.6f} (should be ~0)")
print(f"Scaled data std: {customers_scaled.std():.6f} (should be ~1)")

### Step 2: Fit K-Means with k=3

Let's assume we want to segment customers into 3 groups based on business requirements.

In [ ]:
# Fit K-Means with k=3
kmeans = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=10)
cluster_labels = kmeans.fit_predict(customers_scaled)

# Add cluster labels to original dataframe
customers['Cluster'] = cluster_labels

print(f"K-Means fitted with {kmeans.n_clusters} clusters")
print(f"\nCluster sizes:")
print(customers['Cluster'].value_counts().sort_index())

In [ ]:
# Visualize the clusters
plt.figure(figsize=(10, 6))
colors = ['red', 'blue', 'green']
for cluster in range(3):
    cluster_data = customers[customers['Cluster'] == cluster]
    plt.scatter(cluster_data['Income'], cluster_data['Spending_Score'], 
               c=colors[cluster], label=f'Cluster {cluster}', alpha=0.6, s=60)

# Plot cluster centers (need to inverse transform)
centers_original = scaler.inverse_transform(kmeans.cluster_centers_)
plt.scatter(centers_original[:, 0], centers_original[:, 1], 
           c='black', marker='X', s=300, edgecolors='yellow', linewidths=2, label='Centers')

plt.xlabel('Annual Income ($k)')
plt.ylabel('Spending Score (1-100)')
plt.title('Customer Segmentation with K-Means (k=3)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Step 3: Evaluate Clustering Quality

Two common metrics:
- **Inertia (WCSS)**: Sum of squared distances from points to their cluster centers (lower is better)
- **Silhouette Score**: Measures how similar points are to their own cluster vs. other clusters (-1 to 1, higher is better)

In [ ]:
# Calculate clustering metrics
inertia = kmeans.inertia_
silhouette = silhouette_score(customers_scaled, cluster_labels)

print(f"Inertia (WCSS): {inertia:.2f}")
print(f"Silhouette Score: {silhouette:.4f}")
print("\n💡 Interpretation:")
print(f"   - Silhouette score of {silhouette:.2f} suggests {'good' if silhouette > 0.5 else 'moderate'} cluster separation")
print("   - Scores closer to 1.0 indicate well-separated, compact clusters")

### Step 4: The Elbow Method

How do we choose k? The **elbow method** plots inertia for different k values and looks for an "elbow" where adding more clusters gives diminishing returns.

In [ ]:
# Test different k values
k_range = range(2, 9)
inertias = []
silhouettes = []

for k in k_range:
    kmeans_temp = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    kmeans_temp.fit(customers_scaled)
    inertias.append(kmeans_temp.inertia_)
    silhouettes.append(silhouette_score(customers_scaled, kmeans_temp.labels_))

# Plot elbow curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(k_range, inertias, marker='o', linewidth=2, markersize=8)
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Inertia (WCSS)')
ax1.set_title('Elbow Method: Inertia')
ax1.grid(True, alpha=0.3)

ax2.plot(k_range, silhouettes, marker='o', color='orange', linewidth=2, markersize=8)
ax2.set_xlabel('Number of Clusters (k)')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Scores')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("💡 Look for the 'elbow' where inertia starts decreasing more slowly.")
print(f"   Best silhouette score: k={list(k_range)[np.argmax(silhouettes)]}")

### 🧠 Your Turn — Interpret the Clusters

Now that you've clustered the customers, let's understand what makes each cluster unique.

**Tasks:**
- Calculate the mean income and spending score for each cluster
- Give each cluster a descriptive business name (e.g., "Budget Shoppers", "High Rollers")
- Consider: What marketing strategy would you recommend for each segment?

💡 **Hint:** Use `customers.groupby('Cluster')[['Income', 'Spending_Score']].mean()`

In [ ]:
# Your code here


---

## Section 2: Introduction to Principal Component Analysis (PCA)

**PCA** reduces dimensionality by creating new features (principal components) that are combinations of original features. These components:
- Capture maximum variance in the data
- Are uncorrelated with each other
- Are ordered by importance (PC1 explains most variance, PC2 second-most, etc.)

### Example: Reducing Housing Features

Imagine you have many correlated housing features. PCA can reduce them to a few meaningful components.

In [ ]:
# Create sample housing data with correlated features
np.random.seed(42)
n_houses = 100

# Base size variable
size_factor = np.random.normal(1500, 300, n_houses)

# Create correlated features based on size
housing_data = pd.DataFrame({
    'SqFt': size_factor + np.random.normal(0, 50, n_houses),
    'Bedrooms': (size_factor / 300) + np.random.normal(0, 0.5, n_houses),
    'Bathrooms': (size_factor / 400) + np.random.normal(0, 0.3, n_houses),
    'Garage_Cars': (size_factor / 600) + np.random.normal(0, 0.3, n_houses)
})

print(f"Housing data shape: {housing_data.shape}")
print(f"\nFirst few rows:")
print(housing_data.head())

print(f"\nCorrelation matrix (notice high correlations):")
print(housing_data.corr().round(2))

### Step 1: Standardize Before PCA

PCA is sensitive to scale, so we always standardize first (unless features are already on the same scale).

In [ ]:
# Standardize the housing features
scaler_pca = StandardScaler()
housing_scaled = scaler_pca.fit_transform(housing_data)

print(f"Scaled data shape: {housing_scaled.shape}")
print(f"Mean: {housing_scaled.mean():.6f} (should be ~0)")
print(f"Std: {housing_scaled.std():.6f} (should be ~1)")

### Step 2: Fit PCA

Let's fit PCA with all components to see the full picture.

In [ ]:
# Fit PCA with all components
pca_housing = PCA()
pca_housing.fit(housing_scaled)

print(f"Number of components: {pca_housing.n_components_}")
print(f"\nVariance explained by each component:")
for i, var in enumerate(pca_housing.explained_variance_ratio_, 1):
    print(f"  PC{i}: {var*100:.2f}%")

print(f"\nCumulative variance:")
cumsum = np.cumsum(pca_housing.explained_variance_ratio_)
for i, var in enumerate(cumsum, 1):
    print(f"  First {i} PC(s): {var*100:.2f}%")

### Step 3: Interpret Components with Loadings

**Loadings** show how original features contribute to each principal component. High absolute loadings indicate strong influence.

In [ ]:
# Create loadings dataframe
loadings_df = pd.DataFrame(
    pca_housing.components_.T,
    columns=[f'PC{i+1}' for i in range(pca_housing.n_components_)],
    index=housing_data.columns
)

print("Component Loadings:")
print(loadings_df.round(3))

print("\n💡 Interpretation:")
print("   PC1: All loadings are similar and positive → 'Overall house size'")
print("   PC2: Mixed signs → Captures contrasts between features")

### Step 4: Visualize with Scree Plot

A **scree plot** helps decide how many components to keep.

In [ ]:
# Create scree plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Individual variance
ax1.bar(range(1, len(pca_housing.explained_variance_ratio_) + 1),
        pca_housing.explained_variance_ratio_, alpha=0.7)
ax1.set_xlabel('Principal Component')
ax1.set_ylabel('Variance Explained')
ax1.set_title('Scree Plot')
ax1.grid(True, alpha=0.3, axis='y')

# Cumulative variance
cumulative = np.cumsum(pca_housing.explained_variance_ratio_)
ax2.plot(range(1, len(cumulative) + 1), cumulative, marker='o', linewidth=2)
ax2.axhline(y=0.90, color='r', linestyle='--', label='90% threshold')
ax2.set_xlabel('Number of Components')
ax2.set_ylabel('Cumulative Variance')
ax2.set_title('Cumulative Variance Explained')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"💡 PC1 alone captures {pca_housing.explained_variance_ratio_[0]*100:.1f}% of variance!")
print(f"   This makes sense: these features are highly correlated (all related to size).")

### Step 5: Transform the Data

Finally, we transform our data into the principal component space.

In [ ]:
# Keep just 2 components for demonstration
pca_2 = PCA(n_components=2)
housing_pca = pca_2.fit_transform(housing_scaled)

print(f"Original shape: {housing_scaled.shape} (100 houses × 4 features)")
print(f"Transformed shape: {housing_pca.shape} (100 houses × 2 components)")
print(f"\nVariance retained: {pca_2.explained_variance_ratio_.sum()*100:.1f}%")

# Create dataframe with PC scores
housing_pca_df = pd.DataFrame(housing_pca, columns=['PC1', 'PC2'])
print("\nFirst few rows in PC space:")
print(housing_pca_df.head())

---

## Part B — Independent Challenges (40 minutes)

Now it's your turn to apply K-Means and PCA to two real datasets: the Ames housing data and a set of breast tumor measurements. The specific numerical results you get will be used to answer questions on this week's **Canvas homework quiz**.

* **The support decreases as you go.** Challenges 1–2 give you starter code with blanks (🧩), Challenge 3 gives you an outline only (📝), and Challenge 4 is on your own (🚀).
* **Work together, submit individually.** Talk through the challenges with your group and help each other get unstuck, but **write your own code**. You'll each submit your own notebook and quiz.
* **DO NOT USE AI** to generate code for you.
* Feel free to ask questions or seek help from the instructor. Periodically, we'll stop and walk through parts of a challenge together.
* **Important:** use the exact settings given: `random_state=RANDOM_STATE` in every split, `KMeans`, `PCA` and `LogisticRegression`, `n_init=10` for every `KMeans`, and `test_size=0.2, stratify=y` for the split. These produce the exact numbers needed for the quiz. Steps marked **📝** feed directly into quiz questions.

### Challenge 1 — Segment the Ames Housing Market

🧩 **Starter code provided:** fill in the `___` blanks.

**Business Context:** You're a real estate analyst. Your marketing team wants to split the Ames housing market into **3 segments** so they can tailor their listings and ads to each one.

**Your Task:**
1. **Prepare the data**: Use these six features: `GrLivArea`, `TotalBsmtSF`, `GarageArea`, `YearBuilt`, `OverallQual`, `OverallCond`. Fill any missing `TotalBsmtSF` and `GarageArea` values with 0 (no basement or no garage), then standardize all six.
2. **Choose K**: Fit `KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)` for every k from 2 to 10, recording inertia and silhouette score. 📝 Which k has the highest silhouette score?
3. **Fit the final model** with k=3 (the number of segments marketing asked for), using the same settings. 📝 Record the silhouette score (4 decimals), the inertia (2 decimals), and the number of homes in each cluster.
4. **Profile the clusters**: Compare the average feature values in each cluster, then give each one a segment name a marketer would understand.

In [ ]:
# Step 1: Load the data (PROVIDED), select the features, fill missing values, and standardize
ames = pd.read_csv('https://raw.githubusercontent.com/bradleyboehmke/uc-bana-4080/main/data/ames_clean.csv')
features = ['GrLivArea', 'TotalBsmtSF', 'GarageArea', 'YearBuilt', 'OverallQual', 'OverallCond']

ames_cluster = ames[features].copy()
ames_cluster['TotalBsmtSF'] = ames_cluster['TotalBsmtSF'].fillna(___)
ames_cluster['GarageArea'] = ames_cluster['GarageArea'].fillna(___)

scaler = StandardScaler()
ames_scaled = scaler.___(ames_cluster)
print(f"{ames_scaled.shape[0]} homes, {ames_scaled.shape[1]} features")

In [ ]:
# Step 2: Fit K-Means for every k from 2 to 10, recording inertia and silhouette score
k_values = range(___, ___)
inertias = []
silhouettes = []
for k in k_values:
    kmeans = KMeans(n_clusters=___, n_init=10, random_state=RANDOM_STATE)
    kmeans.fit(___)
    inertias.append(kmeans.___)
    silhouettes.append(silhouette_score(ames_scaled, kmeans.___))

# Plot the elbow and silhouette curves (PROVIDED)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(list(k_values), inertias, marker='o')
ax1.set(xlabel='Number of clusters (k)', ylabel='Inertia', title='Elbow method')
ax2.plot(list(k_values), silhouettes, marker='o', color='darkorange')
ax2.set(xlabel='Number of clusters (k)', ylabel='Silhouette score', title='Silhouette scores')
plt.tight_layout()
plt.show()

best_k = list(k_values)[np.argmax(silhouettes)]
print(f"Highest silhouette score: k={best_k} 📝")

In [ ]:
# Step 3: Fit the final model with k=3
kmeans_final = KMeans(n_clusters=___, n_init=10, random_state=RANDOM_STATE)
ames_cluster['Cluster'] = kmeans_final.fit_predict(___)

print(f"Silhouette score: {silhouette_score(ames_scaled, ames_cluster['Cluster']):.4f} 📝")
print(f"Inertia: {kmeans_final.___:.2f} 📝")
print("\nHomes per cluster 📝")
print(ames_cluster['Cluster'].___().sort_index())

In [ ]:
# Step 4: Profile each cluster by its average feature values
ames_cluster.groupby(___)[features].___().round(1)

*Step 4 — Your segment names (one per cluster, with a sentence on what makes it different):*

### Challenge 2 — Compress 30 Tumor Measurements with PCA

🧩 **Starter code provided:** fill in the `___` blanks.

**Business Context:** You're working with medical researchers who recorded **30 measurements** of each breast tumor (radius, texture, concavity, and so on). Many of the measurements are closely related, which makes the data hard to visualize and explore. You'll use PCA to see how much of the information a few components can hold.

**Your Task:**
1. **Prepare the data**: Separate the 30 measurement columns (`X`) from the `diagnosis` column (`y`, M = malignant, B = benign), then standardize `X`. This challenge explores all 569 patients, so there is no train/test split yet.
2. **Fit PCA** with all components: `PCA()`. 📝 Record the percentage of variance explained by PC1, and by PC1 and PC2 together (2 decimals).
3. **Count components**: 📝 How many components are needed to explain at least 90% of the variance? At least 95%?
4. **Read the loadings**: 📝 Which 5 features have the largest **absolute** loadings on PC1? What do they have in common?
5. **Visualize**: Run the provided scree plot.

In [ ]:
# Step 1: Load the data (PROVIDED), separate the features from the diagnosis, and standardize
breast_cancer = pd.read_csv('https://raw.githubusercontent.com/bradleyboehmke/uc-bana-4080/main/data/breast_cancer.csv')
X = breast_cancer.drop(columns='diagnosis')
y = breast_cancer['diagnosis']

scaler = StandardScaler()
X_scaled = scaler.___(___)
print(f"{X.shape[1]} features, {X.shape[0]} patients ({(y == 'M').sum()} malignant, {(y == 'B').sum()} benign)")

In [ ]:
# Step 2: Fit PCA with all components and compute the share of variance each one explains
pca = PCA()
pca.fit(___)
explained = pca.___               # share of the variance for each component
cumulative = np.___(explained)    # running total

print(f"PC1 explains {explained[0] * 100:.2f}% of the variance 📝")
print(f"PC1 + PC2 explain {cumulative[___] * 100:.2f}% 📝")

# Step 3: How many components to reach at least 90% and at least 95% of the variance?
n_90 = np.argmax(cumulative >= ___) + 1
n_95 = np.argmax(cumulative >= ___) + 1
print(f"Components for 90%: {n_90} 📝")
print(f"Components for 95%: {n_95} 📝")

In [ ]:
# Step 4: Top 5 features by absolute PC1 loading
loadings = pd.DataFrame(
    pca.components_.T,
    index=X.columns,
    columns=[f'PC{i + 1}' for i in range(pca.n_components_)]
)
top5 = loadings['PC1'].___().nlargest(___)
print("Top 5 features on PC1 (absolute loading) 📝")
print(top5.round(3))

*Step 4 — What do PC1's top features have in common? What might a high PC1 score mean for a tumor?*

In [ ]:
# Step 5: Scree plot (PROVIDED)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
components = range(1, len(explained) + 1)
ax1.bar(components, explained)
ax1.set(xlabel='Principal component', ylabel='Share of variance', title='Scree plot')
ax2.plot(components, cumulative, marker='o')
ax2.axhline(0.90, color='darkorange', linestyle='--', label='90%')
ax2.axhline(0.95, color='green', linestyle='--', label='95%')
ax2.set(xlabel='Number of components', ylabel='Cumulative share', title='Cumulative variance explained')
ax2.legend()
plt.tight_layout()
plt.show()

### Challenge 3 — Feed PCA Components to a Classifier

📝 **Outline only:** the code cell lists each step as a comment. You write all the code.

**Business Context:** The researchers want a model that classifies tumors as malignant or benign. A model with fewer inputs is faster and easier to maintain, so they ask: *can PCA components replace the 30 original measurements without hurting accuracy?*

**Your Task:**
1. **Split and scale**: Split `X` and `y` with `test_size=0.2`, `random_state=RANDOM_STATE`, `stratify=y`. Fit a `StandardScaler` on the **training** data only, then transform both the training and test data.
2. **Baseline (all 30 features)**: Train `LogisticRegression(max_iter=10000, random_state=RANDOM_STATE)` on the scaled training data. 📝 Record the test accuracy and the overfitting gap (training accuracy minus test accuracy), both to 4 decimals.
3. **PCA model (95% of the variance)**: Fit `PCA(n_components=0.95, random_state=RANDOM_STATE)` on the scaled **training** data, transform both sets, and train the same logistic regression on the components. 📝 Record how many components were kept, the test accuracy, the overfitting gap, and the dimensionality reduction: `(1 - components / 30) * 100`.
4. **Compare component counts**: Repeat Step 3 with `PCA(n_components=n, random_state=RANDOM_STATE)` for each `n` in `[2, 5, 10, 15, 20, 25]`, recording training and test accuracy. 📝 Which number of components gives the highest test accuracy?
5. **Visualize and reflect**: Plot test accuracy against the number of components, with a horizontal line at the baseline's test accuracy.

💡 **Hint:** PCA is fit like a scaler: `fit_transform()` on the training data, then `transform()` on the test data. Fitting it on the test data leaks information.

In [ ]:
# Step 1: Split X and y (test_size=0.2, random_state=RANDOM_STATE, stratify=y),
#         then fit a StandardScaler on the training data and transform both sets

# Step 2: Baseline logistic regression on all 30 scaled features
#         (max_iter=10000, random_state=RANDOM_STATE); print test accuracy and gap 📝

# Step 3: PCA(n_components=0.95, random_state=RANDOM_STATE): fit on scaled training data,
#         transform both; train the same model; print components kept, test accuracy,
#         gap, and dimensionality reduction 📝

# Step 4: Loop over n in [2, 5, 10, 15, 20, 25]; for each, fit PCA and the model,
#         and record training and test accuracy; print which n has the highest test accuracy 📝

# Step 5: Plot test accuracy vs. number of components, with the baseline as a horizontal line


*Step 5 — Your answers:*

1. Did the 95%-variance PCA model beat the baseline on the test set? Compare the two overfitting gaps. Why might fewer, uncorrelated inputs generalize better?
2. In Step 4 you compared six component counts **on the test set** and picked the best. Based on Week 12, why is that best test accuracy likely optimistic? How would you choose the number of components properly?

### Challenge 4 — Find the Diagnoses Without Labels

🚀 **On your own:** no starter code. Use `X_scaled` and `y` from Challenge 2.

**Business Context:** The researchers ask a bold question: *if we had never recorded a single diagnosis, would the tumor measurements alone separate malignant from benign?* You'll find out by clustering the patients without ever showing K-Means the labels, and only then comparing the clusters with the true diagnoses.

> 🧰 **New tools for this challenge**
>
> * **Majority-label mapping:** K-Means numbers its clusters 0 and 1 in no particular order. To compare clusters with diagnoses, label each cluster with the diagnosis that is **most common among its patients**, then score those labels with `accuracy_score` as usual. `pd.crosstab(clusters, y)` shows you which diagnosis dominates each cluster.
> * **`homogeneity_score(y, clusters)`**: "Is each cluster pure?" It is 1.0 when every cluster contains patients with only one diagnosis.
> * **`completeness_score(y, clusters)`**: "Is each diagnosis kept together?" It is 1.0 when all patients with the same diagnosis land in the same cluster.
>
> Both are already imported in the Setup cell, and both take the true labels first and the cluster labels second.

**Your Task:**
1. **Reduce**: Fit `PCA(n_components=2, random_state=RANDOM_STATE)` on `X_scaled` (all 569 patients, no split) and transform it to 2 components.
2. **Cluster**: Fit `KMeans(n_clusters=2, n_init=10, random_state=RANDOM_STATE)` on the 2 components. 📝 Record the silhouette score of the clusters on the 2-component data (4 decimals).
3. **Compare with the truth**: Map each cluster to its majority diagnosis. 📝 Record the accuracy of those mapped labels against `y` (4 decimals).
4. **Score the clusters**: 📝 Record the homogeneity and completeness scores (4 decimals). Which one is higher, and what does that tell you about the clusters?
5. **Visualize and reflect**: Make two scatter plots of PC1 against PC2, one colored by cluster and one by the true diagnosis. How does this unsupervised accuracy compare with Challenge 3's supervised models, and why would you still use the labels when you have them?

In [ ]:
# Your code here


*Steps 4–5 — Your answers:*

---

## 🎯 Lab Wrap-Up & Reflection

### ✅ What You Accomplished
In this lab, you practiced:
- Applying K-Means clustering to segment data into meaningful groups
- Using the elbow method and silhouette scores to evaluate cluster quality
- Interpreting cluster characteristics for business insights
- Executing the complete PCA workflow from standardization to transformation
- Using scree plots to determine optimal components
- Interpreting component loadings to understand what PCs represent
- Comparing ML models with original vs. PCA-transformed features
- Evaluating tradeoffs between dimensionality reduction and performance

### 🤔 Reflection Questions
Take 2-3 minutes to consider:
- When would you choose clustering over PCA, and vice versa?
- What surprised you about the PCA results? Did reducing dimensions improve or hurt model performance?
- How might you use these techniques in a real data science project?

### 🔗 Connection to Course Goals
Unsupervised learning completes your machine learning toolkit. You now know supervised learning (prediction with labels), unsupervised learning (pattern discovery without labels), and how to combine these techniques effectively. These are foundational skills for any data science role.

### 📋 Next Steps
- **Quiz:** Complete the Week 13 online quiz in Canvas using your recorded answers from Part B
- **Next Week:** Final exam review and course wrap-up
- **Additional Practice:** Try PCA on other high-dimensional datasets (images, text, genomics)

---
**💾 Save your work** and make sure you've recorded all the values needed for the quiz!

## 🚨 Troubleshooting & Common Issues

**Issue 1:** "My silhouette scores don't match the quiz"
- **Solution:** Make sure you're using exactly `random_state=42` and `n_init=10` for all KMeans models. Also verify you filled missing values correctly (TotalBsmtSF and GarageArea with 0).

**Issue 2:** "My PCA variance percentages are slightly different"
- **Solution:** Ensure you standardized the data before PCA. Also round to the exact number of decimals specified (2 decimals for variance %, 4 decimals for accuracy).

**Issue 3:** "My logistic regression accuracies don't match"
- **Solution:** Check three things: (1) `test_size=0.2, random_state=42, stratify=y` in train_test_split, (2) `max_iter=10000, random_state=42` in LogisticRegression, (3) `random_state=42` in PCA.

**Issue 4:** "What if my silhouette score for k=3 is slightly off?"
- **Solution:** The quiz accepts a small range (e.g., 0.2280-0.2300). If you're outside this range, double-check your random_state and n_init parameters.

**General Debugging Tips:**
- Print intermediate results to verify each step
- Check data shapes after each transformation
- Make sure you're fitting on training data and transforming both train and test
- If stuck, review the guided examples in Part A for similar patterns